# Welcome to the Federated Analysis Workshop!

## Contents
1. [Introduction](#1.-Introduction)
2. [Forming the executors](#2.-Forming-the-executors)
    - [Exercise 1: Cohort discovery](#Exercise-1:-Cohort-discovery)
    - [Incidence and prevalence: forming the task](#Incidence-and-prevalence-analysis:-forming-the-task)
    - [Patient profiles: forming the tasks](#Patient-profiles-analysis:-forming-the-tasks)
3. [Submitting and visualising with the Workbench](#3.-Submitting-and-visualising-with-the-Workbench)
    - [Exercise 2: Validate the connection](#Exercise-2:-Validate-the-connection-to-the-Submission-layer)
    - [Incidence and prevalence: submission](#Incidence-and-prevalence-analysis:-submission)
    - [Incidence and prevalence: visualisation](#Incidence-and-prevalence-analysis:-visualisation)
    - [Patient profiles: submission](#Patient-profiles-analysis:-submission)
    - [Patient profiles: results](#Patient-profiles-analysis:-results)
4. [What next?](#4.-What-next?)

## 1. Introduction

This Jupyter notebook is a tutorial for running analyses through Five Safes TES (5S-TES). You will use Python packages, such as the 5S-TES Workbench, to define Darwin EU analyses, send them as tasks to a 5S-TES submission layer, and collect your results.

By the end of this notebook, you will have:
1. Defined two Darwin EU analyses and formed the executors that run them
2. Connected to 5S-TES with the Workbench
3. Submitted the analyses to the TES server as TES messages
4. Fetched the outputs of your analyses and visualised them

### How to use this notebook

A notebook is made of *cells*: blocks of text (like this one) and blocks of Python code. To run a code cell, click on it and press the play button (▶) or `Shift+Enter`.

- Run the cells **in order**, from top to bottom. Later cells depend on earlier ones.
- While a cell is running, `[*]` appears next to it. Wait until it turns into a number before moving on.
- The first line of each code cell is a label: `Cell 0`, `Cell 1`, and so on. The text refers to cells by these labels. (The numbers in square brackets that Jupyter shows are different, and you can ignore them.)
- Text values (called *strings*) must be inside double quotes, e.g. `"Alex"`.
- If you see an error, read its last line and the instructions above the cell, then ask us for help.

Before each code cell, we describe the task and the expected output. The full guide on how to run the Workbench is available [here](https://docs.federated-analytics.ac.uk/tutorials/submit-to-5s-tes/workbench).

### Key words

You have seen these in the presentation. Here is a reminder:

| Term | Meaning |
| --- | --- |
| TES | Task Execution Service: a standard way to describe code that should be run |
| TRE | Trusted Research Environment: a secure place where the data stays and the analysis runs |
| OMOP | A common data model that stores health data in the same format everywhere |
| Concept ID | The number that identifies a medical concept (e.g. a diagnosis) in OMOP |
| Cohort | A group of patients who share a characteristic, e.g. a diabetes diagnosis |
| Incidence | How many *new* cases of a condition appear in a period of time |
| Prevalence | How many people have a condition at a given time |
| Darwin EU | A European network that runs standardised analyses on health data; we use its analysis tools |
| Executor | What a TRE runs: an `image` (the "code engine") plus `commands` |
| Task and bundle | A task is one analysis step; a bundle combines several tasks into one set of commands |

Start by loading the code supporting this notebook. Select Cell 0 below and run it. It loads the libraries and creates a random name for your cohort, so you don't need to change anything.

In [ ]:
# Cell 0: set up the notebook (nothing to change here)
import secrets
import shlex
import string
import warnings
from datetime import date

import matplotlib.pyplot as plt
import pandas as pd
from five_safes_tes_workbench.workbench import Workbench
from i_p_utils import (
    combine_incidence_results,
    load_5s_tes_incidence_result,
    plot_incidence_results,
    plot_jittered,
)
from omop_metadata_utils import DistributionCodesets
from summarised_result import SummarisedResult
from tes_omop_callers.define_cohorts import DefineConceptCohort
from tes_omop_callers.incidence_prevalence import IncidencePrevalence
from tes_omop_callers.patient_profiles import AddCharacteristics, SummariseCohort
from tes_omop_callers.shell_bundlers import CohortShellBundler
from ui import CodesetDisplay

alphabet = string.ascii_lowercase + string.digits
cohort_name = shlex.quote("".join(secrets.choice(alphabet) for _ in range(8)))
image = "ghcr.io/health-informatics-uon/omop-r-tools:sha-07f2d58"

warnings.filterwarnings("ignore")
print("Imported libraries, keep going!")

## 2. Forming the executors

In TES, an executor describes what we want to run on a TRE. It has an `image` (the "code engine") and `commands` (what the code engine should run).

For this workshop, the image has already been chosen, so you only need to define the commands. Commands are hard to type correctly, so we provide functions that write them for you. Each function creates a `task` (one analysis step). A `bundler` combines tasks and turns them into the list of commands for the code engine.

In the next exercises, we will form two bundles, one for each type of analysis: **Incidence Prevalence**, and **Summary of Patient Profiles**. Both are explained in the presentation.

### Exercise 1: Cohort discovery

Before doing an analysis, we need to know which OMOP concept IDs are available and relevant. In this workshop, we are interested in patients with diabetes.

1. Run Cell 1 and type `diabetes` into the search box.
2. Find the two concept IDs on the left-hand side of the chart.
3. In Cell 2, replace the `...` with those two IDs, separated by a comma, e.g. `[111, 222]`.

In [ ]:
# Cell 1: search the concepts available in each TRE
codesets = DistributionCodesets(
    {
        "Nottingham": "metadata/1242/output.json",
        "Singapore": "metadata/Singapore/output.json",
    }
)

codeset_display = CodesetDisplay(codesets)
codeset_display.display()

After replacing the `...`, run Cell 2 to define the cohort we want to analyse. This cohort is used in both analysis bundles.

In [ ]:
# Cell 2: define the diabetes cohort
# Replace the ... with the two concept IDs, e.g. [111, 222]
diabetes_concept_ids = [...]

if ... in diabetes_concept_ids:
    raise ValueError("Please replace the ... with the two concept IDs from Cell 1.")

cohort_task = DefineConceptCohort(
    cohort_name=cohort_name,
    concept_set={cohort_name: diabetes_concept_ids},
)
print("Good job! A new cohort is defined correctly.")

### Incidence and prevalence analysis: forming the task

For the Incidence and Prevalence analysis, we add a task that estimates the incidence of the diabetes cohort defined in Cell 2. Cell 3 creates this task and bundles it with the cohort task from Cell 2. (Your cohort gets a random name, made in Cell 0, so it is different from everyone else's.)

We look at the incidence from 1 January 1984 to 1 January 2004. You can change these dates in `date(year, month, day)`. If you do, run Cell 3 again, and then submit again (Cell 6).

In [ ]:
# Cell 3: form the Incidence and Prevalence bundle
incidence_task = IncidencePrevalence(
    outcome_cohort_name=cohort_name,
    denominator_cohort_name=f"{cohort_name}_denominator",
    denominator_cohort_date_range=(
        date(1984, 1, 1),
        date(2004, 1, 1),
    ),
    estimate_incidence_output_path="outputs/incidence.csv",
)

incidence_bundler = CohortShellBundler(
    image=image,
    tasks=[cohort_task, incidence_task],
)

print("Bundled Incidence Prevalence analysis successfully!")

### Patient profiles analysis: forming the tasks

For the Summary of Patient Profiles analysis, we first need a task that adds characteristics to each patient (such as age and sex). Then we need a task that summarises those characteristics.

There are many characteristics we can add, e.g. age, sex, prior observation, future observation and whether the patient is in observation. ("Observation" is the time period during which a patient's health data is recorded.) This workshop sets default values for them. You can change the values to see how the results change; run Cell 4 again and submit again (Cell 10).

Run Cell 4 to form the task that adds the characteristics, the task that summarises them, and a bundle that combines both with the cohort task from Cell 2.

In [ ]:
# Cell 4: form the Patient Profiles bundle
add_char_task = AddCharacteristics(
    cohort_name=cohort_name,
    index_date="cohort_end_date",
    add_age=True,
    age_group=[(0, 17), (18, 64), (65, 150)],
    missing_month=6,
    missing_day=15,
    age_impose_month=True,
    age_impose_day=True,
    add_sex=True,
    add_prior_observation=True,
    add_future_observation=True,
    add_in_observation=True,
    in_observation_window=[(0, 30)],
    complete_interval=True,
    use_demographics=True,
)

summarise_task = SummariseCohort(
    cohort_table_name=cohort_name,
    variables=["age", "sex", "prior_observation", "future_observation", "in_observation"],
    estimates=["count", "percentage", "median", "q25", "q75"],
)

patient_profiles_bundler = CohortShellBundler(
    image=image,
    tasks=[cohort_task, add_char_task, summarise_task],
)
print("Patient profiles bundler is formed correctly.")

## 3. Submitting and visualising with the Workbench

After forming the executors, we use the Workbench to build the full TES message and submit the tasks to the TES server. The Workbench then retrieves the results from the Submission layer's storage. We also prepared some utilities to visualise the results. Let's do it!

In the code, the two TREs are called `Nottingham TRE 01` (which we call "Nottingham") and `BII-FA-TRE-DEMO` (which we call "Singapore").

### Exercise 2: Validate the connection to the Submission layer

Before submitting the tasks to the TES server, we need to check that we can connect to the Submission layer. At the top of Cell 5, fill in:
- `researcher_name`: your name
- `username` and `password`: the ones we gave you today

Remember that text goes inside double quotes, e.g. `"Alex"`. Then run Cell 5. If everything works, it ends with "You're connecting just fine!".

In [ ]:
# Cell 5: connect to the Submission layer
# Fill in your details (text goes inside double quotes)
researcher_name = ""
username = ""
password = ""

if "" in (researcher_name, username, password):
    raise ValueError("Please fill in researcher_name, username and password at the top of this cell.")

wb = Workbench()
wb.validate(
    project="BIIFedADemo",
    tres=["Nottingham TRE 01", "BII-FA-TRE-DEMO"],
    tes_base_url="https://api.5s-tes.federated-research.com/",
    keycloak_url="https://drs-core-identity.azurewebsites.net/",
    client_id="Dare-Control-S3",
    client_secret="tBqx2MQnMI9oEG1QDKNpq3EKkPZMs7M6",
    username=username,
    password=password,
)

# Test the connection by fetching the outputs of a task that has already run
wb.fetch_outputs(task_id=3677)

print("\n\n\n----------------------------")
print("You're connecting just fine!")

### Incidence and prevalence analysis: submission

Run Cell 6 to build the full TES message for the Incidence and Prevalence analysis and submit it to the TES server.

In [ ]:
# Cell 6: submit the Incidence and Prevalence analysis
wb.build_tes.custom(
    name=f"Incidence Prevalence - {researcher_name}",
    description="Testing the incidence omop caller",
    executors=[incidence_bundler.render_executor()],
    outputs=[
        {
            "name": "Analysis Results Location",
            "description": "Analysis Results Location",
            "url": "s3://",
            "path": "/outputs",
            "type": "DIRECTORY",
        }
    ],
)

wb.submit()

After submitting, you can check the status of your task on [the Submission layer](https://5s-tes.federated-research.com/Submission/GetAllSubmissions) or run Cell 7, which checks the task's status before fetching the outputs. This can take a while, so we will wait for everyone here.

In [ ]:
# Cell 7: fetch the Incidence and Prevalence outputs
incidence_output = wb.fetch_outputs()

### Incidence and prevalence analysis: visualisation

Run Cell 8 to read the outputs into the notebook, reshape them for analysis and plot the results of each TRE.

Run Cell 9 to plot the combined result and compare it with the individual TREs.

In [ ]:
# Cell 8: plot the result of each TRE

# Read the outputs into the notebook and wrangle them into the right shape for analysis
incidence_results = load_5s_tes_incidence_result(incidence_output)

# Plot the results
plot_incidence_results(incidence_results)

In [ ]:
# Cell 9: combine the TREs and compare
overall_incidence_result = combine_incidence_results(list(incidence_results.values()))

overall_incidence_result.plot_incidence()

fig, ax = plt.subplots(figsize=(12, 8))

plot_jittered(incidence_results["Nottingham TRE 01"], ax=ax)
plot_jittered(incidence_results["BII-FA-TRE-DEMO"], offset_weeks=-8, ax=ax)
overall_incidence_result.plot_incidence(axes=ax)

ax.legend(["Nottingham", "Singapore", "Combined"])

plt.show()

### Patient profiles analysis: submission

Run Cell 10 to build the full TES message for the Patient Profiles analysis and submit it to the TES server.

In [ ]:
# Cell 10: submit the Patient Profiles analysis
wb.build_tes.custom(
    name=f"Patient Profiles - {researcher_name}",
    description="Testing the patientprofiles omop caller",
    executors=[patient_profiles_bundler.render_executor()],
    outputs=[
        {
            "name": "Analysis Results Location",
            "description": "Analysis Results Location",
            "url": "s3://",
            "path": "/outputs",
            "type": "DIRECTORY",
        }
    ],
)

wb.submit()

After submitting, you can check the status of your task on [the Submission layer](https://5s-tes.federated-research.com/Submission/GetAllSubmissions) or run Cell 11, which checks the task's status before fetching the outputs. This can take a while, so we will wait for everyone here.

In [ ]:
# Cell 11: fetch the Patient Profiles outputs
characteristics_results = wb.fetch_outputs()

### Patient profiles analysis: results

Run Cell 12 to see the summary of the patient profiles from each TRE, side by side. You don't need to understand the code in this cell.

In [ ]:
# Cell 12: summarise the patient profiles of each TRE (nothing to change here)
characteristics_data = {
    k: SummarisedResult(pd.read_csv(v[0]))
    for k, v in characteristics_results.items()
}


def group_tres(results):
    dfs = []
    for tre, result in results.items():
        df = result.data
        df["TRE"] = tre
        dfs.append(df)
    all_tres = pd.concat(dfs)
    level_cols = ["TRE", "group_level", "strata_level", "variable_level"]
    all_tres = all_tres.loc[~(df.cdm_name.isna())]

    return all_tres.pivot(
        index=level_cols,
        columns=["variable_name", "estimate_name"],
        values="estimate_value",
    ).T.style.highlight_null(props="color: transparent;")


group_tres(characteristics_data)

## 4. What next?

Congratulations! By running this notebook, you have learned how to:
- search for concepts and define a cohort across several TREs
- form analysis tasks with the Darwin EU tools and submit them with the 5S-TES Workbench
- collect the results from each TRE and combine them, without moving any patient-level data

Things to think about:
- Why can we combine the results from both TREs without seeing the patient data?
- How do the results differ between Nottingham and Singapore, and why might that be?

If you want to learn more, check out the [workbench documentation](https://docs.federated-analytics.ac.uk/tutorials/submit-to-5s-tes/workbench), [other notebooks](https://github.com/Health-Informatics-UoN/5s-TES-notebooks/), and [our packages](https://github.com/Health-Informatics-UoN/omop-r-toolkit/) using the Darwin-EU tools.